# STITCH & STRING (Human) — KG Processing
**Project:** MetaboGlue / EvoAge KG &nbsp;|&nbsp; **Species:** *Homo sapiens*  
**Notebook:** `STITCH_for_EvoKG_Processing_human.ipynb` 

**Outputs:**

| Section | Relation | Output file |
|---------|----------|-------------|
| §4 — STITCH Chem–Chem | ChemicalEntity_ChemicalEntity | `Chemical_Chemical_STITCH.csv` |
| §5 — STITCH Chem–Protein | ChemicalEntity_Protein | `Chemical_Protein_STITCH.csv` |
| §6 — STRING Protein–Protein | Protein_Protein | `Protein_Protein_STRING.csv` |

**Filter:** `experimental > 0` in all three sections.




## 0. Path Configuration

In [1]:
import os

# ── Edit only these five lines ───────────────────────────────────────────────
BASE_PATH    = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
DB_BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/databases_for_mapping/"
STITCH_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/stitch/"   # STITCH v5.0 TSV files
STRING_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/string/"   # STRING v12.0 TXT file
STITCH_OUT_PATH = "/storage/Arushi//090526_EvoAge/kg_formation/processed_data/stitch/"      # all output CSVs written here
STRING_OUT_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/string/hsap/"      # all output CSVs written here
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(STITCH_OUT_PATH, exist_ok=True)
os.makedirs(STRING_OUT_PATH, exist_ok=True)
print("STITCH_PATH  :", STITCH_PATH)
print("STRING_PATH  :", STRING_PATH)
print("DB_BASE_PATH :", DB_BASE_PATH)
print("STITCH_OUT_PATH     :", STITCH_OUT_PATH)
print("STRING_OUT_PATH     :", STRING_OUT_PATH)

STITCH_PATH  : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/stitch/
STRING_PATH  : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/string/
DB_BASE_PATH : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/databases_for_mapping/
STITCH_OUT_PATH     : /storage/Arushi//090526_EvoAge/kg_formation/processed_data/stitch/
STRING_OUT_PATH     : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/string/hsap/


## 1. Imports

In [2]:
import os
import re
import numpy as np
import pandas as pd
from glob import glob


## 2. Shared Helpers

In [3]:
KG_DESIRED_COLS = [
    "Head", "Relation", "Tail",
    "Head_type", "Tail_type",
    "KG_Source",
    "Head_detail_name", "Head_SMILE_name",
    "Tail_detail_name",
    "Head_ID_IS", "Tail_ID_IS",
]

def reorder(df):
    """Return df keeping only KG_DESIRED_COLS that exist, in order."""
    return df[[c for c in KG_DESIRED_COLS if c in df.columns]]


### 2.1 `clean_chemical_id()`


In [4]:
def clean_chemical_id(chem_id):
    """
    Convert a STITCH chemical ID to a plain PubChem CID string.
    Handles both CIDs (stereospecific) and CIDm (merged) prefixes.

    Examples:
        'CIDs00000100' -> '100'
        'CIDm00012340' -> '12340'

    """
    return re.sub(r'^CID[sm]', '', str(chem_id)).lstrip('0') or '0'


## 3. Reference Dictionaries

### 3.1 Ensembl ENSP → UniProt AC


In [5]:
Uniprot_file = pd.read_csv(
    os.path.join(DB_BASE_PATH, "ENSEMBL", "Homo_sapiens.GRCh38.113.uniprot.tsv"),
    sep="\t"
)


trembl    = Uniprot_file[Uniprot_file["db_name"] == "Uniprot/SPTREMBL"]
swissprot = Uniprot_file[Uniprot_file["db_name"] == "Uniprot/SWISSPROT"]

Uniprot_file_dict = {
    **dict(zip(trembl["protein_stable_id"],    trembl["xref"])),
    **dict(zip(swissprot["protein_stable_id"], swissprot["xref"])),
}
del Uniprot_file, trembl, swissprot
print(f"Uniprot_file_dict (ENSP → AC): {len(Uniprot_file_dict):,} entries")


Uniprot_file_dict (ENSP → AC): 120,162 entries


### 3.2 UniProt AC → RecName

In [6]:
Uniprot_Recname = pd.read_csv(
    os.path.join(DB_BASE_PATH, "uniprot", "Uniprot_ID_Recname.csv")
)
Uniprot_Recname_dict = dict(zip(Uniprot_Recname["AC"], Uniprot_Recname["RecName"]))
del Uniprot_Recname
print(f"Uniprot_Recname_dict (AC → RecName): {len(Uniprot_Recname_dict):,} entries")


Uniprot_Recname_dict (AC → RecName): 794,151 entries


### 3.3 PubChem CID → IUPAC Name / SMILES


In [7]:
Pubchem = pd.read_pickle(os.path.join(DB_BASE_PATH, "pubchem", "combined_df.pkl"))

Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem["PUBCHEM_COMPOUND_CID"], Pubchem["PUBCHEM_IUPAC_NAME"]))
Pubchem_CID_Smile_Dict = dict(zip(Pubchem["PUBCHEM_COMPOUND_CID"], Pubchem["PUBCHEM_SMILES"]))


del Pubchem
print(f"Pubchem_IUPAC_CID_Dict : {len(Pubchem_IUPAC_CID_Dict):,}")
print(f"Pubchem_CID_Smile_Dict : {len(Pubchem_CID_Smile_Dict):,}")


Pubchem_IUPAC_CID_Dict : 119,177,440
Pubchem_CID_Smile_Dict : 119,177,440


## 4. STITCH — Chemical–Chemical
**Source:** `chemical_chemical.links.detailed.v5.0.tsv`  
**Filter:** `experimental > 0`



In [8]:
CC = pd.read_csv(
    os.path.join(STITCH_PATH, "hsap","chemical_chemical.links.detailed.v5.0.tsv"),
    sep="\t"
)
print(f"Raw rows: {len(CC):,}")

# Inspect ID prefix format (informational)
prefix_sample = set(CC["chemical1"].apply(lambda x: x[:4]))
print(f"Chemical ID prefixes found: {prefix_sample}")

CC = CC[CC["experimental"] > 0]
print(f"After experimental > 0 filter: {len(CC):,}")


Raw rows: 17,705,818
Chemical ID prefixes found: {'CIDs', 'CIDm'}
After experimental > 0 filter: 747,992


In [9]:

CC["chemical1"] = CC["chemical1"].map(clean_chemical_id)
CC["chemical2"] = CC["chemical2"].map(clean_chemical_id)

CC = CC.rename(columns={"chemical1": "Head", "chemical2": "Tail"})

CC["Head_detail_name"] = CC["Head"].map(Pubchem_IUPAC_CID_Dict)
CC["Tail_detail_name"] = CC["Tail"].map(Pubchem_IUPAC_CID_Dict)


CC["Head_SMILE_name"] = CC["Head"].map(Pubchem_CID_Smile_Dict)

CC = CC[~CC["Head_detail_name"].isna()]
CC = CC[~CC["Tail_detail_name"].isna()]

CC["Head_ID_IS"] = "Pubchem"
CC["Tail_ID_IS"] = "Pubchem"
CC["Head_type"]  = "ChemicalEntity"
CC["Tail_type"]  = "ChemicalEntity"
CC["Relation"]   = CC["Head_type"] + "_" + CC["Tail_type"]
CC["KG_Source"]  = "STITCH"


CC = reorder(CC)
print(f"Final Chemical–Chemical rows: {len(CC):,}")
CC.head(3)


Final Chemical–Chemical rows: 716,016


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Head_detail_name,Head_SMILE_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
39840,91392180,ChemicalEntity_ChemicalEntity,10071196,ChemicalEntity,ChemicalEntity,STITCH,"6,6-dimethyl-11-prop-2-ynoxy-5,6,6a,7-tetrahyd...",C[N+]1(CCC2=C3C1CC4=C(C3=CC=C2)C(=CC=C4)OCC#C)C,1-[(4-fluorophenyl)methyl]-1-(1-methylpiperidi...,Pubchem,Pubchem
39841,91392180,ChemicalEntity_ChemicalEntity,107992,ChemicalEntity,ChemicalEntity,STITCH,"6,6-dimethyl-11-prop-2-ynoxy-5,6,6a,7-tetrahyd...",C[N+]1(CCC2=C3C1CC4=C(C3=CC=C2)C(=CC=C4)OCC#C)C,2-chloro-6-piperazin-1-ylpyrazine,Pubchem,Pubchem
39842,91392180,ChemicalEntity_ChemicalEntity,11237985,ChemicalEntity,ChemicalEntity,STITCH,"6,6-dimethyl-11-prop-2-ynoxy-5,6,6a,7-tetrahyd...",C[N+]1(CCC2=C3C1CC4=C(C3=CC=C2)C(=CC=C4)OCC#C)C,"[(3R)-1-azabicyclo[2.2.2]octan-3-yl] 6-[(3S,4R...",Pubchem,Pubchem


In [10]:

CC.to_csv(os.path.join(STITCH_OUT_PATH, "stitch_HUMAN_Chemical_Chemical_STITCH.csv"), index=False)
print("Saved: Chemical_Chemical_STITCH.csv")

del CC

Saved: Chemical_Chemical_STITCH.csv


## 5. STITCH — Chemical–Protein
**Source:** `protein_chemical.links.detailed.v5.0.tsv`  
**Filter:** `experimental > 0`  
**Protein ID chain:** `9606.ENSP... → ENSP... → UniProt AC → RecName`



In [11]:
CP = pd.read_csv(
    os.path.join(STITCH_PATH, "hsap","protein_chemical.links.detailed.v5.0.tsv"),
    sep="\t"
)
print(f"Raw rows: {len(CP):,}")
print("Columns:", list(CP.columns))

CP = CP[CP["experimental"] > 0]
print(f"After experimental > 0 filter: {len(CP):,}")


Raw rows: 15,473,939
Columns: ['chemical', 'protein', 'experimental', 'prediction', 'database', 'textmining', 'combined_score']
After experimental > 0 filter: 8,842,952


In [12]:

CP["Head"] = CP["chemical"].map(clean_chemical_id)

# Strip species prefix from protein column
CP["Tail"] = CP["protein"].str.replace(r"^9606\.", "", regex=True)

# Annotate Head (chemical)
CP["Head_detail_name"] = CP["Head"].map(Pubchem_IUPAC_CID_Dict)

CP["Head_SMILE_name"]  = CP["Head"].map(Pubchem_CID_Smile_Dict)

# Map ENSP → UniProt AC → RecName (two-step protein chain)
CP["Tail_ID"] = CP["Tail"].map(Uniprot_file_dict)
CP = CP[~CP["Tail_ID"].isna()]
print(f"After ENSP→UniProt mapping: {len(CP):,}")

CP["Tail_detail_name"] = CP["Tail_ID"].map(Uniprot_Recname_dict)
CP = CP[~CP["Tail_detail_name"].isna()]
print(f"After RecName annotation: {len(CP):,}")

# Swap: Tail ← UniProt AC; original ENSP stored in Tail_ID
CP[["Tail", "Tail_ID"]] = CP[["Tail_ID", "Tail"]]

CP = CP[~CP["Head_detail_name"].isna()]

CP["Head_ID_IS"] = "Pubchem"
CP["Tail_ID_IS"] = "Uniprot_ID"
CP["Head_type"]  = "ChemicalEntity"
CP["Tail_type"]  = "Protein"
CP["Relation"]   = CP["Head_type"] + "_" + CP["Tail_type"]
CP["KG_Source"]  = "STITCH"


CP = reorder(CP)
print(f"Final Chemical–Protein rows: {len(CP):,}")
CP.head(3)


After ENSP→UniProt mapping: 8,192,011
After RecName annotation: 8,050,949
Final Chemical–Protein rows: 7,843,748


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Head_detail_name,Head_SMILE_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
3525,91668531,ChemicalEntity_Protein,P21917,ChemicalEntity,Protein,STITCH,"N-[4-(1-oxo-3,4-dihydro-2H-pyrido[4,3-b]indol-...",CC(=O)NCCCCN1C2=C(C3=CC=CC=C31)C(=O)NCC2,D(4) dopamine receptor,Pubchem,Uniprot_ID
3526,91668531,ChemicalEntity_Protein,P46663,ChemicalEntity,Protein,STITCH,"N-[4-(1-oxo-3,4-dihydro-2H-pyrido[4,3-b]indol-...",CC(=O)NCCCCN1C2=C(C3=CC=CC=C31)C(=O)NCC2,B1 bradykinin receptor,Pubchem,Uniprot_ID
3527,91668531,ChemicalEntity_Protein,Q8NFJ6,ChemicalEntity,Protein,STITCH,"N-[4-(1-oxo-3,4-dihydro-2H-pyrido[4,3-b]indol-...",CC(=O)NCCCCN1C2=C(C3=CC=CC=C31)C(=O)NCC2,Prokineticin receptor 2,Pubchem,Uniprot_ID


In [13]:
CP.to_csv(os.path.join(STITCH_OUT_PATH, "stitch_HUMAN_Chemical_Protein_STITCH.csv"), index=False)
print("Saved: Chemical_Protein_STITCH.csv")

del CP

Saved: Chemical_Protein_STITCH.csv


## 6. STRING — Protein–Protein (Human)
**Source:** `9606.protein.links.detailed.v12.0.txt` (whitespace-separated)  
**Filter:** `experimental > 0`  
**Protein ID chain:** `9606.ENSP... → ENSP... → UniProt AC → RecName`



In [14]:
PP = pd.read_csv(
    os.path.join(STRING_PATH,"hsap", "9606.protein.links.detailed.v12.0.txt"),
    sep=r"\s+", engine="python"
)
print(f"Raw rows: {len(PP):,}")

PP = PP[PP["experimental"] > 0]
print(f"After experimental > 0 filter: {len(PP):,}")


Raw rows: 13,715,404
After experimental > 0 filter: 5,847,852


In [15]:
# Strip species prefix
PP["protein1"] = PP["protein1"].str.replace(r"^9606\.", "", regex=True)
PP["protein2"] = PP["protein2"].str.replace(r"^9606\.", "", regex=True)

PP = PP.rename(columns={"protein1": "Head", "protein2": "Tail"})

# Map ENSP → UniProt AC
PP["Head_ID"] = PP["Head"].map(Uniprot_file_dict)
PP["Tail_ID"] = PP["Tail"].map(Uniprot_file_dict)
PP = PP[~PP["Head_ID"].isna() & ~PP["Tail_ID"].isna()]
print(f"After ENSP→UniProt mapping: {len(PP):,}")

# Map UniProt AC → RecName
PP["Head_detail_name"] = PP["Head_ID"].map(Uniprot_Recname_dict)
PP["Tail_detail_name"] = PP["Tail_ID"].map(Uniprot_Recname_dict)
PP = PP[~PP["Head_detail_name"].isna() & ~PP["Tail_detail_name"].isna()]
print(f"After RecName annotation: {len(PP):,}")

# Swap: UniProt AC → Head/Tail; original ENSP saved in Head_ID/Tail_ID
PP[["Head", "Head_ID"]] = PP[["Head_ID", "Head"]]
PP[["Tail", "Tail_ID"]] = PP[["Tail_ID", "Tail"]]


PP["Head_ID_IS"] = "Uniprot_ID"
PP["Tail_ID_IS"] = "Uniprot_ID"
PP["Head_type"]  = "Protein"
PP["Tail_type"]  = "Protein"
PP["Relation"]   = PP["Head_type"] + "_" + PP["Tail_type"]
PP["KG_Source"]  = "STRING"


PP = reorder(PP)
print(f"Final Protein–Protein rows: {len(PP):,}")
PP.head(3)


After ENSP→UniProt mapping: 5,618,700
After RecName annotation: 5,101,030
Final Protein–Protein rows: 5,101,030


,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS
0,P84085,Protein_Protein,Q86X27,Protein,Protein,STRING,ADP-ribosylation factor 5,Ras-specific guanine nucleotide-releasing fact...,Uniprot_ID,Uniprot_ID
1,P84085,Protein_Protein,Q9C0D6,Protein,Protein,STRING,ADP-ribosylation factor 5,FH2 domain-containing protein 1,Uniprot_ID,Uniprot_ID
2,P84085,Protein_Protein,P36543,Protein,Protein,STRING,ADP-ribosylation factor 5,V-type proton ATPase subunit E 1,Uniprot_ID,Uniprot_ID


In [16]:

PP.to_csv(os.path.join(STRING_OUT_PATH, "Protein_Protein_STRING.csv"), index=False)
print("Saved: Protein_Protein_STRING.csv")
del PP


Saved: Protein_Protein_STRING.csv


## 7. Summary Audit


In [17]:
cols_audit = ["Relation", "Head_type", "Tail_type", "KG_Source", "Head_ID_IS", "Tail_ID_IS"]
rows = []

for fp in sorted(glob(os.path.join(STITCH_OUT_PATH, "*STITCH.csv"))):
    try:
        tmp = pd.read_csv(fp)
        row = {"File": os.path.basename(fp), "Triples": len(tmp)}
        for col in cols_audit:
            if col in tmp.columns:
                row[col] = set(tmp[col].dropna().unique())
        rows.append(row)
    except Exception as e:
        print(f"  Error: {fp} — {e}")

for fp in sorted(glob(os.path.join(STRING_OUT_PATH, "*.csv"))):
    try:
        tmp = pd.read_csv(fp)
        row = {"File": os.path.basename(fp), "Triples": len(tmp)}
        for col in cols_audit:
            if col in tmp.columns:
                row[col] = set(tmp[col].dropna().unique())
        rows.append(row)
    except Exception as e:
        print(f"  Error: {fp} — {e}")

audit_df = pd.concat([pd.DataFrame([r]) for r in rows], ignore_index=True)
display(audit_df)
print(f"\nTotal triples: {audit_df['Triples'].sum():,}")


,File,Triples,Relation,Head_type,Tail_type,KG_Source,Head_ID_IS,Tail_ID_IS
0,stitch_HUMAN_Chemical_Chemical_STITCH.csv,716016,{ChemicalEntity_ChemicalEntity},{ChemicalEntity},{ChemicalEntity},{STITCH},{Pubchem},{Pubchem}
1,stitch_HUMAN_Chemical_Protein_STITCH.csv,7843748,{ChemicalEntity_Protein},{ChemicalEntity},{Protein},{STITCH},{Pubchem},{Uniprot_ID}
2,stitch_HUMAN__Chemical_Chemical_STITCH.csv,716016,{ChemicalEntity_ChemicalEntity},{ChemicalEntity},{ChemicalEntity},{STITCH},{Pubchem},{Pubchem}
3,Protein_Protein_STRING.csv,5101030,{Protein_Protein},{Protein},{Protein},{STRING},{Uniprot_ID},{Uniprot_ID}



Total triples: 14,376,810
